# MetroLab · 02 — Tabla de modelado

Reemplaza `03_TABLA_MODELADO` (paso 9). Entrada: `v3/panel_5min` (cuaderno 01). Salida: `v3/tabla_modelo`, una fila por segmento × calzada × ventana **con dato**, partida por `fecha`.

| Paso | Qué agrega |
|---|---|
| 9.2 | Calendario (hora, día, festivos, puentes, Semana Santa, días de pago) |
| 9.3 | Historia del segmento: velocidad hace 5, 10, 15, 30 y 60 min, ayer y hace 7 días |
| 9.4 | Vecinos: tramos de 500 m adelante y atrás, distancia al próximo trancón |
| 9.5 | Eventos: cierres en el segmento y a menos de 1 km adelante |
| 9.6 | Objetivos: velocidad suavizada dentro de 15, 30 y 60 min |

**Modos**
- `"completo"`: arma toda la tabla, mes por mes (la primera vez, después de correr el 01 en modo completo).
- `"actualizar"`: arma solo las fechas que dejó pendientes el cuaderno 01 **y el día anterior** (para completar los objetivos de sus últimas horas). Usa 7 días de historia antes, para los rezagos.

Cambios frente a la versión anterior:
- La grilla segmento × ventana se **arma completa** aquí; así los rezagos (`lag`) son correctos aunque falte un día entero de datos.
- Los eventos salen en la misma lectura del panel (antes se leía dos veces).
- La columna `conjunto` (entrenamiento / validación / prueba) ya no se guarda: la define el cuaderno 03 al reentrenar.

## 0. Parámetros, Spark y Azure

In [ ]:
# ============================================================
# 0.1 Parámetros de la corrida
# ============================================================
MODO = "actualizar"     # "completo" | "actualizar"
assert MODO in ("completo", "actualizar")
print("Modo:", MODO)

In [ ]:
# ============================================================
# 0.2 Spark y Azure (misma configuración de 00_Conexion_Azure_PySpark)
# ============================================================
import os, sys, subprocess, time, json, shutil, gc, glob
import importlib.metadata
try:
    import pyspark
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark"], check=True)
    import pyspark
import psutil

# ---- 1. Entorno: Colab (con Google Drive) o GitHub Actions ----
try:
    from google.colab import userdata
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

def secreto(*nombres):
    """Primer secreto que exista: Secretos de Colab (llave) o variables de entorno (GitHub Actions)."""
    for n in nombres:
        try:
            v = userdata.get(n) if EN_COLAB else os.environ.get(n)
        except Exception:
            v = None
        if v:
            return v
    raise ValueError(f"Falta el secreto {nombres[0]} (revisa la llave de Secretos de Colab)")

CONTENT = "/content" if EN_COLAB else os.path.abspath("trabajo")
os.makedirs(CONTENT, exist_ok=True)

PROJECT_DIR = None                               # carpeta del proyecto en Google Drive
if EN_COLAB:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/Waze_GD"
    if os.path.isdir(PROJECT_DIR):
        PATH_NOTEBOOKS, PATH_SALIDAS, PATH_CONFIG = (f"{PROJECT_DIR}/{c}" for c in ("Notebooks", "Salidas", "Config"))
        os.makedirs(PATH_SALIDAS, exist_ok=True)
        print("✅ Google Drive conectado · proyecto:", PROJECT_DIR)
    else:
        print("⚠️ No encontré", PROJECT_DIR, "(se sigue sin Drive)")
        PROJECT_DIR = None

# ---- 2. Recursos (Colab Pro: 8 CPU y ~51 GB → 8 núcleos y ~30 GB para Spark) ----
CPU_SPARK = os.cpu_count()
RAM_GB = psutil.virtual_memory().total / 1024**3
MEMORIA_SPARK = f"{max(4, int(RAM_GB * 0.6))}g"
PARTICIONES_SHUFFLE = CPU_SPARK * 8               # 8 CPU → 64, como en el script de conexión
TMP_LOCAL = f"{CONTENT}/tmp_metrolab"             # temporales en el disco local (mucho más rápido que Azure)
os.makedirs(TMP_LOCAL, exist_ok=True)

# ---- 3. Conector Hadoop-Azure ----
HADOOP_VERSION = "3.4.1"
AZURE_PACKAGE = f"org.apache.hadoop:hadoop-azure:{HADOOP_VERSION}"
os.environ["PYSPARK_SUBMIT_ARGS"] = f"--packages {AZURE_PACKAGE} pyspark-shell"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"        # evita conflictos de red de Spark en Colab
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# ---- 4. Spark: reutiliza la sesión si ya existe ----
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F, types as T
import numpy as np
import pandas as pd

spark = SparkSession.getActiveSession()
if spark is not None:
    print("✅ Se reutiliza la sesión Spark existente")
else:
    spark = (
        SparkSession.builder
        .appName("MetroLab_02_Tabla")
        .master(f"local[{CPU_SPARK}]")
        .config("spark.driver.memory", MEMORIA_SPARK)
        .config("spark.driver.maxResultSize", "2g")
        .config("spark.sql.shuffle.partitions", str(PARTICIONES_SHUFFLE))
        .config("spark.default.parallelism", str(CPU_SPARK * 2))
        .config("spark.sql.adaptive.enabled", "true")
        .config("spark.local.dir", f"{TMP_LOCAL}/spark")
        .getOrCreate()
    )
    print("✅ Nueva sesión Spark iniciada")
spark.sparkContext.setLogLevel("WARN")

# Ajustes de la línea MetroLab (se aplican también si la sesión ya existía)
for k, v in {
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    # Un solo reloj: 'ts_s' y 'v' son HORA DE BOGOTÁ guardada como si fuera UTC;
    # con la sesión en UTC, Spark muestra esas horas tal cual, sin correrlas 5 horas
    "spark.sql.session.timeZone": "UTC",
    "spark.sql.sources.partitionOverwriteMode": "dynamic",              # reescribe solo las fechas tocadas
    "spark.sql.sources.partitionColumnTypeInference.enabled": "false",  # 'fecha' se lee como texto
    "spark.sql.execution.arrow.pyspark.enabled": "true",
}.items():
    spark.conf.set(k, v)

# ---- 5. Autenticación Azure ADLS Gen2 ----
account    = secreto("AZURE_STORAGE_ACCOUNT", "storage_account_name")
key        = secreto("AZURE_STORAGE_KEY", "storage_account_key")
filesystem = secreto("AZURE_FILE_SYSTEM", "file_system_name")
endpoint   = f"{account}.dfs.core.windows.net"
hconf = spark.sparkContext._jsc.hadoopConfiguration()
hconf.set(f"fs.azure.account.auth.type.{endpoint}", "SharedKey")
hconf.set(f"fs.azure.account.key.{endpoint}", key)
del key

BASE      = f"abfss://{filesystem}@{endpoint}/Waze_feed_tvt_parquet"
RUTA_JAMS = f"{BASE}/salidas_intermedias/02_ASIGNACION_ESPACIAL/jams_autosur_json"
SALIDA    = f"{BASE}/salidas_intermedias/03_MODELACION"
V3        = f"{SALIDA}/v3"                   # todo lo nuevo queda aquí; lo anterior no se toca

OFFSET_S  = 5 * 3600     # Bogotá = UTC − 5 (Colombia no tiene horario de verano)
VENTANA_S = 300          # ventanas de 5 minutos

# ---- Utilidades de archivos ----
Path = spark._jvm.org.apache.hadoop.fs.Path
def fs_de(ruta):  return Path(ruta).getFileSystem(hconf)
def existe(ruta): return fs_de(ruta).exists(Path(ruta))
def borrar(ruta):
    if existe(ruta):
        fs_de(ruta).delete(Path(ruta), True)
def local(nombre): return f"file://{TMP_LOCAL}/{nombre}"
def guardar_local(df, nombre):
    """Escribe en el disco de Colab y vuelve a leer: corta el plan de Spark y evita recalcular."""
    df.write.mode("overwrite").parquet(local(nombre))
    return spark.read.parquet(local(nombre))
def fechas_de(ruta):
    """Fechas guardadas en una tabla partida por 'fecha' (solo lista carpetas, no lee datos)."""
    if not existe(ruta):
        return []
    return sorted(s.getPath().getName().split("=", 1)[1] for s in fs_de(ruta).listStatus(Path(ruta))
                  if s.isDirectory() and s.getPath().getName().startswith("fecha="))
def fecha_de(col_segundos):
    return F.date_format(F.timestamp_seconds(col_segundos), "yyyy-MM-dd")

# ---- 6. Validación de conexión y rutas ----
print("\n=== CONFIGURACIÓN SPARK ===")
print("Versión:", spark.version, "· Master:", spark.sparkContext.master,
      "· Memoria:", spark.sparkContext.getConf().get("spark.driver.memory", MEMORIA_SPARK),
      "· Particiones:", spark.conf.get("spark.sql.shuffle.partitions"))
print("\n=== VALIDACIÓN DE CONEXIÓN AZURE ===")
rutas_ok = {}
for nombre, ruta in [("Waze_feed_tvt_parquet", BASE), ("02_ASIGNACION_ESPACIAL/jams_autosur_json", RUTA_JAMS),
                     ("03_MODELACION", SALIDA)]:
    rutas_ok[nombre] = existe(ruta)
    print(("✅ OK" if rutas_ok[nombre] else "❌ NO EXISTE"), "|", nombre)
if not all(rutas_ok.values()):
    raise FileNotFoundError("Revisa las rutas que no existen (o las credenciales de Azure)")
print("✅ CONEXIÓN AZURE VALIDADA · salidas nuevas en:", V3)

In [ ]:
# ============================================================
# 0.3 Qué fechas se arman
# ============================================================
RUTA_PANEL = f"{V3}/panel_5min"
RUTA_TM    = f"{V3}/tabla_modelo"
RUTA_PEND  = f"{V3}/control/pendientes_tabla"

FECHAS_PANEL = fechas_de(RUTA_PANEL)
if not FECHAS_PANEL:
    raise SystemExit("No hay panel en v3: corre primero el cuaderno 01")

if MODO == "completo":
    borrar(RUTA_TM)
    meses = sorted({f[:7] for f in FECHAS_PANEL})
    TANDAS = [[f for f in FECHAS_PANEL if f.startswith(m)] for m in meses]      # mes por mes
else:
    pend = sorted(r.fecha for r in spark.read.schema("fecha string").parquet(RUTA_PEND).collect()) if existe(RUTA_PEND) else []
    if not pend:
        raise SystemExit("No hay fechas pendientes: la tabla ya está al día.")
    desde = str((pd.Timestamp(pend[0]) - pd.Timedelta(days=1)).date())      # + el día anterior
    TANDAS = [[f for f in FECHAS_PANEL if desde <= f <= pend[-1]]]

print(f"Panel: {len(FECHAS_PANEL)} fechas ({FECHAS_PANEL[0]} a {FECHAS_PANEL[-1]})")
print(f"Tandas a armar: {len(TANDAS)} · fechas: {sum(len(t) for t in TANDAS)} "
      f"({TANDAS[0][0]} a {TANDAS[-1][-1]})")

## 1. Funciones de los pasos 9.2 a 9.6

Son las mismas reglas de `03_TABLA_MODELADO`. El calendario lee los segundos `v` **tal cual** (ya son hora de Bogotá).

In [ ]:
# ============================================================
# 1.1 Calendario (9.2)
# ============================================================
try:
    import holidays
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "holidays"], check=True)
    import holidays
from dateutil.easter import easter

VACACIONES_ESCOLARES = []        # opcional: [("2026-06-22", "2026-07-10"), ...]

def calendario(v_ini, v_fin):
    v = np.arange(v_ini, v_fin + VENTANA_S, VENTANA_S)
    t = pd.to_datetime(v, unit="s")                          # sin cambio de zona: ya es hora de Bogotá
    cal = pd.DataFrame({"v": v})
    cal["ventana_local"] = t.strftime("%Y-%m-%d %H:%M")
    cal["fecha"]         = t.strftime("%Y-%m-%d")
    cal["hora"]          = t.hour
    cal["minuto"]        = t.minute
    cal["hora_decimal"]  = t.hour + t.minute / 60
    cal["dia_semana"]    = t.dayofweek
    cal["fin_de_semana"] = cal.dia_semana >= 5
    cal["mes_num"]       = t.month
    cal["dia_mes"]       = t.day
    cal["semana_anio"]   = t.isocalendar().week.to_numpy().astype(int)
    cal["franja"] = pd.cut(cal.hora_decimal, bins=[0, 5, 9, 16, 20, 24], right=False,
                           labels=["madrugada", "pico_am", "valle", "pico_pm", "noche"]).astype(str)

    dias = pd.date_range(t.min().normalize() - pd.Timedelta(days=3), t.max().normalize() + pd.Timedelta(days=3))
    fest = holidays.Colombia(years=sorted(set(dias.year)), language="es")
    d = pd.Series(t.date, index=cal.index)
    cal["festivo"]   = d.map(lambda x: x in fest)
    cal["dia_habil"] = ~cal.fin_de_semana & ~cal.festivo
    puente, retorno = set(), set()
    for dd in dias.date:
        if dd not in fest:
            continue
        dow = pd.Timestamp(dd).dayofweek
        if dow == 0:
            puente  |= {dd, dd - pd.Timedelta(days=1), dd - pd.Timedelta(days=2)}
            retorno |= {dd}
        elif dow == 4:
            puente  |= {dd, dd + pd.Timedelta(days=1), dd + pd.Timedelta(days=2)}
            retorno |= {dd + pd.Timedelta(days=2)}
    cal["puente"]          = d.isin(puente)
    cal["retorno_puente"]  = d.isin(retorno)
    cal["vispera_festivo"] = d.map(lambda x: (pd.Timestamp(x) + pd.Timedelta(days=1)).date() in fest)
    santa = set()
    for a in sorted(set(dias.year)):
        santa |= {easter(a) - pd.Timedelta(days=k) for k in range(8)}
    cal["semana_santa"] = d.isin(santa)
    cal["dia_pago"]     = cal.dia_mes.isin([14, 15, 16, 29, 30, 31, 1])
    vac = set()
    for a, b in VACACIONES_ESCOLARES:
        vac |= set(pd.date_range(a, b).date)
    cal["vacaciones_escolares"] = d.isin(vac)
    cal["hora_sin"] = np.sin(2 * np.pi * cal.hora_decimal / 24)
    cal["hora_cos"] = np.cos(2 * np.pi * cal.hora_decimal / 24)
    cal["dia_sin"]  = np.sin(2 * np.pi * cal.dia_semana / 7)
    cal["dia_cos"]  = np.cos(2 * np.pi * cal.dia_semana / 7)
    return spark.createDataFrame(cal)

# ============================================================
# 1.2 Panel con grilla completa (base de 9.3–9.5)
# ============================================================
vl_seg = spark.read.parquet(f"{V3}/vel_libre_segmento").filter("activo")
SEGS = vl_seg.select("sentido", "FROM_TO", "orden_seg", "abs_ini_m", "longitud_m",
                     "vel_libre_ref_kmh", "vl_respaldo", "vel_libre_mediana_kmh")
COLS_PANEL = ["sentido", "FROM_TO", "v", "estado", "vel_kmh", "cobertura", "retraso_s",
              "cierre_persistente", "cierre_puntual", "hay_peaje", "nivel_max", "n_capturas", "estado_dato"]

def segundos(fecha):
    return int(pd.Timestamp(fecha).value // 10**9)

def leer_panel(v_desde, v_hasta):
    """Todas las ventanas entre dos instantes para los 597 segmentos; donde el panel no tiene fila, 'sin dato'."""
    f0 = str(pd.to_datetime(v_desde, unit="s").date())
    f1 = str(pd.to_datetime(v_hasta, unit="s").date())
    fechas = [f for f in FECHAS_PANEL if f0 <= f <= f1]
    p = (spark.read.parquet(RUTA_PANEL).filter(F.col("fecha").isin(fechas))
              .filter(F.col("v").between(v_desde, v_hasta)).select(*COLS_PANEL))
    ventanas = spark.range(v_desde, v_hasta + VENTANA_S, VENTANA_S).withColumnRenamed("id", "v")
    return (SEGS.crossJoin(F.broadcast(ventanas))
                .join(p, ["sentido", "FROM_TO", "v"], "left")
                .withColumn("estado", F.coalesce(F.col("estado"), F.lit("sin dato"))))

# ============================================================
# 1.3 Historia del segmento (9.3)
# ============================================================
REZAGOS = {"vel_l1": 1, "vel_l2": 2, "vel_l3": 3, "vel_l6": 6, "vel_l12": 12,
           "vel_ayer": 288, "vel_semana": 2016}
MARGEN_ATRAS_S    = max(REZAGOS.values()) * VENTANA_S       # 7 días
MARGEN_ADELANTE_S = 24 * VENTANA_S                          # 2 horas (objetivos a 60 min + suavizado)

w   = Window.partitionBy("sentido", "FROM_TO").orderBy("v")
w30 = w.rowsBetween(-5, 0)

def agregar_historia(df):
    for nombre, k in REZAGOS.items():
        df = df.withColumn(nombre, F.lag("vel_kmh", k).over(w))
    return (df.withColumn("vel_media_30", F.avg("vel_kmh").over(w30))
              .withColumn("vel_min_30",   F.min("vel_kmh").over(w30))
              .withColumn("delta_15",     F.col("vel_kmh") - F.col("vel_l3"))
              .withColumn("indice_vel",   F.col("vel_kmh") / F.col("vel_libre_ref_kmh")))

# ============================================================
# 1.4 Vecinos (9.4)
# ============================================================
TRAMO_M = 500
SIN_TRANCON_M = 10_000

def agregar_vecinos(df):
    df = (df.withColumn("abs_fin_m", F.col("abs_ini_m") + F.col("longitud_m"))
            .withColumn("tramo", F.floor((F.col("abs_ini_m") + F.col("longitud_m") / 2) / TRAMO_M).cast("int")))
    valido = F.col("vel_kmh").isNotNull() & (F.col("vel_kmh") > 0) & (F.col("longitud_m") > 0)
    w_tramo = Window.partitionBy("sentido", "tramo").orderBy("v")
    tr = (df.groupBy("sentido", "tramo", "v")
            .agg(F.sum(F.when(valido, F.col("longitud_m"))).alias("L"),
                 F.sum(F.when(valido, F.col("longitud_m") / (F.col("vel_kmh") / 3.6))).alias("T"),
                 F.sum(F.when(valido, F.col("cobertura") * F.col("longitud_m"))).alias("LC"))
            .withColumn("vel_tramo", F.when((F.col("L") > 0) & (F.col("T") > 0), F.col("L") / F.col("T") * 3.6))
            .withColumn("cob_tramo", F.when(F.col("L") > 0, F.col("LC") / F.col("L")))
            .withColumn("vel_tramo_l3", F.lag("vel_tramo", 3).over(w_tramo))
            .select("sentido", "tramo", "v", "vel_tramo", "cob_tramo", "vel_tramo_l3"))
    tr = guardar_local(tr, "tramos")      # se usa 3 veces: se calcula una sola
    adelante  = tr.select("sentido", (F.col("tramo") - 1).alias("tramo"), "v",
                          F.col("vel_tramo").alias("vel_adelante_500"), F.col("cob_tramo").alias("cob_adelante_500"),
                          F.col("vel_tramo_l3").alias("vel_adelante_l3"))
    adelante2 = tr.select("sentido", (F.col("tramo") - 2).alias("tramo"), "v",
                          F.col("vel_tramo").alias("vel_adelante_1000"))
    atras     = tr.select("sentido", (F.col("tramo") + 1).alias("tramo"), "v",
                          F.col("vel_tramo").alias("vel_atras_500"), F.col("cob_tramo").alias("cob_atras_500"))
    df = (df.join(adelante, ["sentido", "tramo", "v"], "left")
            .join(adelante2, ["sentido", "tramo", "v"], "left")
            .join(atras, ["sentido", "tramo", "v"], "left"))
    w_adel = Window.partitionBy("sentido", "v").orderBy("abs_ini_m").rowsBetween(1, Window.unboundedFollowing)
    ini_tr = F.first(F.when(F.col("estado") == "trancon", F.col("abs_ini_m")), ignorenulls=True).over(w_adel)
    return (df.withColumn("ini_trancon_adelante", ini_tr)
              .withColumn("dist_trancon_adelante_m",
                          F.when(F.col("estado") == "sin dato", None)
                           .when(F.col("ini_trancon_adelante").isNull(), F.lit(float(SIN_TRANCON_M)))
                           .otherwise(F.greatest(F.col("ini_trancon_adelante") - F.col("abs_fin_m"), F.lit(0.0))))
              .drop("ini_trancon_adelante"))

# ============================================================
# 1.5 Eventos (9.5) y objetivos (9.6)
# ============================================================
def agregar_eventos(df):
    hay_cierre = F.greatest(F.coalesce(F.col("cierre_persistente"), F.lit(0)),
                            F.coalesce(F.col("cierre_puntual"), F.lit(0)))
    tr_c = df.groupBy("sentido", "tramo", "v").agg(F.max(hay_cierre).alias("c"))
    for k in [0, 1, 2]:
        t = tr_c.select("sentido", (F.col("tramo") - k).alias("tramo"), "v", F.col("c").alias(f"c{k}"))
        df = df.join(t, ["sentido", "tramo", "v"], "left")
    return (df.withColumn("cierre_cerca_1km", F.greatest(*[F.coalesce(F.col(f"c{k}"), F.lit(0)) for k in [0, 1, 2]]))
              .drop("c0", "c1", "c2")
              .withColumn("segmento_peaje", (F.col("vel_libre_mediana_kmh") < 20).cast("int"))
              .drop("vel_libre_mediana_kmh"))

HORIZONTES = {"15": 3, "30": 6, "60": 12}
def agregar_objetivos(df):
    for h, k in HORIZONTES.items():
        df = (df.withColumn(f"y_{h}",   F.lead("vel_kmh", k).over(w))
                .withColumn(f"y_{h}_s", F.avg("vel_kmh").over(w.rowsBetween(k - 1, k + 1))))
    return df

print("Funciones listas")

## 2. Armar y guardar la tabla

Cada tanda lee el panel con 7 días de margen atrás y 2 horas adelante, calcula todo, y guarda **solo sus fechas** y solo las filas con dato (trancón o fluido). Etapa 1 (historia y vecinos) va al disco de Colab, no a Azure.

In [ ]:
# ============================================================
# 2. Tabla de modelado
# ============================================================
v_max_panel = None
for i, fechas in enumerate(TANDAS, 1):
    t0 = time.time()
    ini = segundos(fechas[0])
    fin = segundos(fechas[-1]) + 86400 - VENTANA_S
    desde, hasta = ini - MARGEN_ATRAS_S, fin + MARGEN_ADELANTE_S

    e0 = guardar_local(agregar_historia(leer_panel(desde, hasta)), "etapa0")    # historia (9.3)
    e1 = guardar_local(agregar_vecinos(e0), "etapa1")                          # vecinos (9.4)
    cal = calendario(ini, fin)
    tabla = (agregar_objetivos(agregar_eventos(e1))
               .filter(F.col("v").between(ini, fin) & F.col("estado").isin("trancon", "fluido"))
               .join(F.broadcast(cal), "v", "inner")
               .drop("abs_fin_m"))
    tabla.repartition("fecha").write.mode("overwrite").partitionBy("fecha").parquet(RUTA_TM)
    print(f"Tanda {i}/{len(TANDAS)}: {fechas[0]} a {fechas[-1]} guardada ({(time.time() - t0) / 60:.1f} min)")

# Ya no hay pendientes
spark.createDataFrame([], "fecha string").write.mode("overwrite").parquet(RUTA_PEND)
print("\nTabla de modelado al día.")

## 3. Revisión rápida

In [ ]:
# ============================================================
# 3. Revisión de las últimas fechas
# ============================================================
ult = fechas_de(RUTA_TM)[-3:]
tm = spark.read.parquet(RUTA_TM).filter(F.col("fecha").isin(ult))
print("Columnas:", len(tm.columns))
(tm.groupBy("fecha")
   .agg(F.count("*").alias("filas"),
        F.round(100 * F.avg((F.col("estado") == "trancon").cast("int")), 2).alias("pct_trancon"),
        F.min("ventana_local").alias("desde"), F.max("ventana_local").alias("hasta"),
        F.round(100 * F.avg(F.col("vel_l1").isNull().cast("int")), 2).alias("pct_sin_l1"),
        F.round(100 * F.avg(F.col("y_30_s").isNull().cast("int")), 2).alias("pct_sin_y30"))
   .orderBy("fecha").show(truncate=False))
print("pct_sin_y30 es alto solo en la última fecha (sus últimas horas aún no tienen futuro).")

for nombre in os.listdir(TMP_LOCAL):
    if nombre != "spark":
        shutil.rmtree(f"{TMP_LOCAL}/{nombre}", ignore_errors=True)
print("Listo. Sigue el cuaderno 03_MODELO.")